# Bias Analysis

This notebook assesses potential bias in NovaCred's past loan approval decisions. The analysis focused in differences in protected features (gender and age), intersectional effects (age × gender), and possible proxy discrimination through features such as ZIP code.

Fairness is measured using loan approval rate comparisons, Disparate Impact (four-fifths rule), Demographic Parity Difference, and chi-square statistical significance tests. The aim is to evaluate  whether observed approval outcomes show systemic disparities.

## Table of Contents

## 1. Executive Summary

## 2. Methodology

## 3. Dataset Overview for Bias Analysis

### 3.1 Importing Necessary Libraries

The analysis relies on the following Python libraries:

- **pandas** – data manipulation, aggregation, and creation of summary tables.
- **numpy** – numerical operations and handling numeric outputs from statistical tests.
- **matplotlib** – basic data visualization.
- **seaborn** – improved statistical visualizations (bar plots).
- **scipy.stats** – statistical testing, specifically the Chi-Square Test of Independence.
- **fairlearn** – computation of fairness metrics such as Demographic Parity Difference.
- **datetime** – calculating applicant age from date of birth.

In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency
from fairlearn.metrics import demographic_parity_difference

from datetime import datetime


### 3.2 Dataset loading

In [2]:
df = pd.read_csv("../data/credit_applications_clean.csv")
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,age
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23.0,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,22.0
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51.0,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,31.0
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41.0,0.21,37909,True,NaN,vacation,3.7,59000.0,34.0
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70.0,0.35,0,True,NaN,NaN,4.3,34000.0,40.0
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15 00:00:00+00:00,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14.0,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,24.0


### 3.3 Dataset Structure and Size

In [5]:
# Checking basic info and distributions
df.info()
df.describe()
df["decision.loan_approved"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 20 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               501 non-null    object 
 1   spending_behavior                 501 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.full_name          501 non-null    object 
 4   applicant_info.email              494 non-null    object 
 5   applicant_info.ssn                496 non-null    object 
 6   applicant_info.ip_address         496 non-null    object 
 7   applicant_info.gender             498 non-null    object 
 8   applicant_info.date_of_birth      496 non-null    object 
 9   applicant_info.zip_code           499 non-null    float64
 10  financials.annual_income          496 non-null    float64
 11  financials.credit_history_months  499 non-null    float64
 12  financia

decision.loan_approved
True     292
False    209
Name: count, dtype: int64

### 3.4 Key variables for the analysis

The bias analysis focuses on the following key variables:

- **decision.loan_approved** — binary outcome indicating whether the loan was approved.
- **applicant_info.gender** — gender of the applicant.
- **applicant_info.date_of_birth / applicant_info_age** — used to derive age and age groups.
- **applicant_info.zip_code** — geographic attribute evaluated as a potential proxy variable.
- **financial variables** (income, credit history, debt-to-income) — used to understand potential structural differences across groups.

### 3.5 Missing Data Checks

In [ ]:
df[["applicant_info.gender","decision.loan_approved"]].isna().sum()

applicant_info.gender     4
decision.loan_approved    0
dtype: int64